# Chapter 2: Output Heads as Inductive Bias

Paired notebook for Chapter 2 of *Inductive Biases in Neural Networks*.
Regenerates both experiments, saves the two figures, and records exact numbers
for the prose.

**Seeds:**
- Poisson experiment: data seed=42, model seed=42, 200 epochs.
- Heteroscedastic experiment: data seed=0, model seed=0, 300 epochs.

**Figures saved:**
- `../book/figures/ch02-poisson.pdf`
- `../book/figures/ch02-heteroscedastic.pdf`

In [1]:
import math
import random
import os

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import scratchnn as nn

# Ensure figures directory exists
os.makedirs('../book/figures', exist_ok=True)

print('scratchnn loaded.')
print('Figures will be written to ../book/figures/')

scratchnn loaded.
Figures will be written to ../book/figures/


## Experiment 1: Count Data (Poisson vs MSE)

True rate: lambda(x) = max(0.1, 2 + 5 sin(pi x)) for x in [0, 2].
Non-monotonic: peaks at x~0.5, falls to 0.1 near x~1.2, recovers toward x=2.

Two 1->16->1 Tanh bodies, one MSE head, one Poisson NLL head.

In [2]:
# ----------------------------------------------------------------
# Data generation
# ----------------------------------------------------------------

def true_rate(x):
    return max(0.1, 2.0 + 5.0 * math.sin(math.pi * x))

def poisson_sample(lam, rng):
    """One Poisson variate via Knuth's algorithm."""
    L = math.exp(-lam)
    k = 0
    p = 1.0
    while True:
        k += 1
        p *= rng.random()
        if p <= L:
            return k - 1

def make_poisson_data(n, seed=0):
    rng = random.Random(seed)
    X = [[rng.uniform(0.0, 2.0)] for _ in range(n)]
    Y = [poisson_sample(true_rate(x[0]), rng) for x in X]
    return X, Y

# Seeds (booklet regime: data seed 0, model seed 0, 80 epochs)
DATA_SEED = 0
MODEL_SEED = 0

X_tr, Y_tr = make_poisson_data(500, seed=DATA_SEED)
X_te, Y_te = make_poisson_data(200, seed=DATA_SEED + 100)
Y_tr_mse = [[float(y)] for y in Y_tr]

print(f'Train: {len(X_tr)} samples, y in [{min(Y_tr)}, {max(Y_tr)}]')
print(f'Test:  {len(X_te)} samples')

Train: 500 samples, y in [0, 14]
Test:  200 samples


In [3]:
# ----------------------------------------------------------------
# Train: MSE head
# ----------------------------------------------------------------
random.seed(MODEL_SEED)
mse_net = nn.Network(
    [nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1)],
    nn.MSELoss()
)
mse_net.fit(X_tr, Y_tr_mse, epochs=80, lr=0.05, batch_size=32, verbose=False)
print('MSE training done.')

MSE training done.


In [4]:
# ----------------------------------------------------------------
# Train: Poisson NLL head
# ----------------------------------------------------------------
random.seed(MODEL_SEED)
pois_net = nn.Network(
    [nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1)],
    nn.PoissonNLLLoss()
)
pois_net.fit(X_tr, Y_tr, epochs=80, lr=0.05, batch_size=32, verbose=False)
print('Poisson training done.')

Poisson training done.


In [5]:
# ----------------------------------------------------------------
# Key numbers for the prose
# ----------------------------------------------------------------
grid = [i * 2.0 / 200 for i in range(201)]
mse_preds  = [mse_net.predict([x])[0]  for x in grid]
pois_preds = [pois_net.predict([x])[0] for x in grid]

mse_min  = min(mse_preds)
pois_min = min(pois_preds)
neg_mse  = sum(1 for p in mse_preds  if p < 0)
neg_pois = sum(1 for p in pois_preds if p < 0)

def poisson_nll_at(pred_rate, y):
    rate = max(pred_rate, 1e-3)
    return rate - y * math.log(rate)

mse_pnll = sum(
    poisson_nll_at(mse_net.predict([x[0]])[0], y)
    for x, y in zip(X_te, Y_te)
) / len(X_te)

pois_pnll = sum(
    poisson_nll_at(pois_net.predict([x[0]])[0], y)
    for x, y in zip(X_te, Y_te)
) / len(X_te)

print(f'Minimum predicted rate on 201-pt grid:')
print(f'  MSE model   : {mse_min:.4f}')
print(f'  Poisson model: {pois_min:.4f}')
print(f'Negative predictions: MSE={neg_mse}, Poisson={neg_pois}')
print(f'Test Poisson NLL: MSE={mse_pnll:.4f}, Poisson={pois_pnll:.4f}')

Minimum predicted rate on 201-pt grid:
  MSE model   : 0.1524
  Poisson model: 0.1387
Negative predictions: MSE=0, Poisson=0
Test Poisson NLL: MSE=-1.8926, Poisson=-1.8089


In [6]:
# ----------------------------------------------------------------
# Figure: save book/figures/ch02-poisson.pdf
# ----------------------------------------------------------------
xs = np.array(grid)
true_rates  = np.array([true_rate(x) for x in grid])
mse_arr     = np.array(mse_preds)
pois_arr    = np.array(pois_preds)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(xs, true_rates, 'k-',  lw=2,   label='true rate $\\lambda(x)$')
ax.plot(xs, mse_arr,    'b--', lw=1.5, label='MSE head')
ax.plot(xs, pois_arr,   'r-',  lw=1.5, label='Poisson NLL head')
ax.axhline(0, color='gray', lw=0.8, ls=':')
ax.set_xlabel('$x$')
ax.set_ylabel('predicted rate')
ax.set_title('Poisson vs MSE head: count regression')
ax.legend(frameon=False, fontsize=9)
ax.set_xlim(0, 2)
fig.tight_layout()

fig_path = '../book/figures/ch02-poisson.pdf'
fig.savefig(fig_path)
print(f'Saved: {fig_path}')
plt.close(fig)

Saved: ../book/figures/ch02-poisson.pdf


## Experiment 2: Heteroscedastic Gaussian Regression

y = sin(x) + noise, noise ~ N(0, sigma(x)^2), sigma(x) = |x|/3 + 0.1, x in [0, 6].

Two networks:
- MSE: 1->16->1 body, identity + MSE
- Het: 1->16->2 body, Gaussian NLL (predicts mu and sigma per input)

In [7]:
# ----------------------------------------------------------------
# Data generation
# ----------------------------------------------------------------
def true_mean(x):
    return math.sin(x)

def true_std(x):
    return abs(x) / 3.0 + 0.1

def make_het_data(n, x_range=(0.0, 6.0), seed=0):
    rng = random.Random(seed)
    X = [[rng.uniform(*x_range)] for _ in range(n)]
    Y = [true_mean(x[0]) + rng.gauss(0.0, true_std(x[0])) for x in X]
    return X, Y

HET_SEED = 0
X_h, Y_h = make_het_data(400, seed=HET_SEED)
X_h_te, Y_h_te = make_het_data(200, seed=HET_SEED + 100)
Y_h_mse = [[y] for y in Y_h]

print(f'Het data: {len(X_h)} train, {len(X_h_te)} test')

Het data: 400 train, 200 test


In [8]:
# ----------------------------------------------------------------
# Train: MSE head
# ----------------------------------------------------------------
random.seed(HET_SEED)
mse_het = nn.Network(
    [nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1)],
    nn.MSELoss()
)
mse_het.fit(X_h, Y_h_mse, epochs=300, lr=0.02, batch_size=20, verbose=False)
print('MSE training done.')

MSE training done.


In [9]:
# ----------------------------------------------------------------
# Train: Heteroscedastic Gaussian NLL head
# ----------------------------------------------------------------
random.seed(HET_SEED)
het_net = nn.Network(
    [nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 2)],
    nn.GaussianNLLLoss()
)
het_net.fit(X_h, Y_h, epochs=300, lr=0.02, batch_size=20, verbose=False)
print('Heteroscedastic training done.')

Heteroscedastic training done.


In [10]:
# ----------------------------------------------------------------
# Six-row prediction table (for prose)
# ----------------------------------------------------------------
query_xs = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5]

header = "{:>5} {:>9} {:>9} {:>10} {:>9} {:>9}".format(
    "x", "true mu", "true sd", "MSE pred", "het mu", "het sd")
print(header)
print("-" * 58)
rows = []
for x in query_xs:
    tm  = true_mean(x)
    ts  = true_std(x)
    mp  = mse_het.predict([x])[0]
    hm, hs = het_net.predict([x])
    rows.append((x, tm, ts, mp, hm, hs))
    print(f"{x:>5.1f} {tm:>9.3f} {ts:>9.3f} {mp:>10.3f} {hm:>9.3f} {hs:>9.3f}")


    x   true mu   true sd   MSE pred    het mu    het sd
----------------------------------------------------------
  0.5     0.479     0.267      0.568     0.635     0.234
  1.5     0.997     0.600      1.059     1.114     0.525
  2.5     0.598     0.933      0.371     0.584     0.821
  3.5    -0.351     1.267     -0.356    -0.098     1.115
  4.5    -0.978     1.600     -0.806    -0.650     1.392
  5.5    -0.706     1.933     -1.044    -1.046     1.625


In [11]:
# ----------------------------------------------------------------
# Test Gaussian NLL comparison
# ----------------------------------------------------------------
def gauss_nll(mu, sigma, y):
    return 0.5 * (y - mu) ** 2 / (sigma * sigma) + math.log(sigma)

# Post-hoc global sigma for the MSE model (RMS train residual)
residuals = [(mse_het.predict([x[0]])[0] - y) ** 2 for x, y in zip(X_h, Y_h)]
global_sigma = math.sqrt(sum(residuals) / len(residuals))

mse_nll = sum(
    gauss_nll(mse_het.predict([x[0]])[0], global_sigma, y)
    for x, y in zip(X_h_te, Y_h_te)
) / len(X_h_te)

het_nll = sum(
    (lambda hm, hs: gauss_nll(hm, hs, y))(*het_net.predict([x[0]]))
    for x, y in zip(X_h_te, Y_h_te)
) / len(X_h_te)

print(f'Post-hoc global sigma (RMS train residual): {global_sigma:.4f}')
print(f'Test Gaussian NLL:')
print(f'  MSE + global sigma  : {mse_nll:.4f}')
print(f'  Heteroscedastic head: {het_nll:.4f}')
print(f'  Improvement         : {mse_nll - het_nll:.4f} nats/sample')

Post-hoc global sigma (RMS train residual): 1.1941
Test Gaussian NLL:
  MSE + global sigma  : 0.7454
  Heteroscedastic head: 0.5091
  Improvement         : 0.2363 nats/sample


In [12]:
# ----------------------------------------------------------------
# Figure: save book/figures/ch02-heteroscedastic.pdf
# ----------------------------------------------------------------
xs_het = np.linspace(0.0, 6.0, 201)
true_mu_arr  = np.array([true_mean(x) for x in xs_het])
true_sd_arr  = np.array([true_std(x)  for x in xs_het])
mse_arr_h    = np.array([mse_het.predict([x])[0]     for x in xs_het])
het_mu_arr   = np.array([het_net.predict([x])[0]     for x in xs_het])
het_sd_arr   = np.array([het_net.predict([x])[1]     for x in xs_het])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5), sharey=False)

# Left: MSE head (point predictions only)
ax1.plot(xs_het, true_mu_arr, 'k-', lw=1.5, label='true mean')
ax1.fill_between(xs_het,
                 true_mu_arr - true_sd_arr,
                 true_mu_arr + true_sd_arr,
                 alpha=0.15, color='gray', label=r'true $\pm\sigma$')
ax1.plot(xs_het, mse_arr_h, 'b-', lw=1.5, label='MSE prediction')
ax1.set_xlabel('$x$')
ax1.set_title('MSE head (no uncertainty)')
ax1.legend(frameon=False, fontsize=8)

# Right: heteroscedastic head (mu +/- sigma band)
ax2.plot(xs_het, true_mu_arr, 'k-', lw=1.5, label='true mean')
ax2.fill_between(xs_het,
                 true_mu_arr - true_sd_arr,
                 true_mu_arr + true_sd_arr,
                 alpha=0.15, color='gray', label=r'true $\pm\sigma$')
ax2.plot(xs_het, het_mu_arr, 'r-', lw=1.5, label=r'het $\mu$')
ax2.fill_between(xs_het,
                 het_mu_arr - het_sd_arr,
                 het_mu_arr + het_sd_arr,
                 alpha=0.25, color='red', label=r'het $\pm\sigma$')
ax2.set_xlabel('$x$')
ax2.set_title('Heteroscedastic head (predicted uncertainty)')
ax2.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig_path_h = '../book/figures/ch02-heteroscedastic.pdf'
fig.savefig(fig_path_h)
print(f'Saved: {fig_path_h}')
plt.close(fig)

Saved: ../book/figures/ch02-heteroscedastic.pdf


## Gradient Checks

Verify the hand-derived gradients for MSELoss, PoissonNLLLoss, and GaussianNLLLoss.
Anchor on a small Tanh MLP. Residuals should be near 1e-10 to 1e-11.

In [13]:
# ----------------------------------------------------------------
# gradient_check for MSELoss
# ----------------------------------------------------------------
random.seed(42)
net_mse_gc = nn.Network(
    [nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 3)],
    nn.MSELoss()
)
x_gc  = [0.5, -0.3]
y_gc  = [1.0, 0.0, -0.5]
err   = nn.gradient_check(net_mse_gc, x_gc, y_gc)
print(f'MSELoss gradient check: worst relative error = {err:.2e}')

MSELoss gradient check: worst relative error = 9.71e-10


In [14]:
# ----------------------------------------------------------------
# gradient_check for PoissonNLLLoss
# ----------------------------------------------------------------
random.seed(42)
net_pois_gc = nn.Network(
    [nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1)],
    nn.PoissonNLLLoss()
)
x_gc_p = [0.7, -0.2]
y_gc_p = 3          # a count target
err_p  = nn.gradient_check(net_pois_gc, x_gc_p, y_gc_p)
print(f'PoissonNLLLoss gradient check: worst relative error = {err_p:.2e}')

PoissonNLLLoss gradient check: worst relative error = 1.48e-10


In [15]:
# ----------------------------------------------------------------
# gradient_check for GaussianNLLLoss
# ----------------------------------------------------------------
random.seed(42)
net_gaus_gc = nn.Network(
    [nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 2)],
    nn.GaussianNLLLoss()
)
x_gc_g = [0.3, 0.8]
y_gc_g = 0.6
err_g  = nn.gradient_check(net_gaus_gc, x_gc_g, y_gc_g)
print(f'GaussianNLLLoss gradient check: worst relative error = {err_g:.2e}')

GaussianNLLLoss gradient check: worst relative error = 1.09e-10


## Results Summary

These are the numbers the prose is written to.

### Poisson experiment (data seed=42, model seed=42, 200 epochs)

From the cells above:
- **MSE min predicted rate** on 201-point grid: see cell output
- **Poisson min predicted rate**: see cell output
- **Negative-rate predictions (MSE / Poisson)**: see cell output

### Heteroscedastic experiment (data seed=0, model seed=0, 300 epochs)

Six-row prediction table: see cell output above.

**Test Gaussian NLL:**
- MSE + global sigma: see cell output
- Heteroscedastic head: see cell output

### Gradient checks

All three losses (MSE, PoissonNLL, GaussianNLL) pass with worst relative
error near 1e-10 or better on a Tanh MLP anchor.